````markdown
# 🎯 Speech + Video Modality — Interview Cheat Sheet

## 1. SPEECH MODALITY

### 1.1 Core Pipeline

```text
Raw Audio
   ↓
Waveform
   ↓
Sampling + Bit Depth
   ↓
STFT / FFT
   ↓
Power Spectrogram
   ↓
Mel Filterbank
   ↓
Log-Mel Spectrogram
   ↓
Whisper Encoder
   ↓
Whisper Decoder
   ↓
Text / Translation
````

### Key idea

> A model never "hears" audio — it consumes a tensor.

| Representation    | Typical Shape         | Purpose                       |
| ----------------- | --------------------- | ----------------------------- |
| Waveform          | `(samples,)`          | Raw audio                     |
| Power spectrogram | `(1+n_fft/2, frames)` | Time-frequency representation |
| Log-mel           | `(80, frames)`        | Whisper encoder input         |
| Token IDs         | `(seq_len,)`          | Whisper decoder               |
| CLAP embedding    | `(512,)`              | Audio/text retrieval          |

---

## 2. Sampling Rate + Nyquist

### Nyquist theorem

For sampling frequency `fs`:

```text
Maximum representable frequency = fs / 2
```

Example:

```text
fs = 16 kHz
Nyquist frequency = 8 kHz
```

16 kHz is commonly used for speech because most speech intelligibility is below
~4 kHz, with important consonant energy extending toward ~8 kHz.

### Important interview point

When downsampling:

```text
Original signal
      ↓
Anti-alias low-pass filter
      ↓
Downsample
```

Never simply do:

```python
y[::k]
```

without filtering, because frequencies above the new Nyquist limit can alias
into lower frequencies.

---

## 3. Bit Depth

Bit depth controls **amplitude resolution**, not bandwidth.

Approximate quantization SNR:

```text
~6 dB improvement per additional bit
```

Example:

```text
16-bit → ~96 dB theoretical dynamic range
```

### Remember

```text
Sample rate → frequency/bandwidth
Bit depth   → amplitude resolution/dynamic range
```

---

# 4. Speech Anatomy

Speech consists roughly of:

```text
Voiced   → quasi-periodic signal
Unvoiced → noise-like signal
```

### Source-filter model

```text
Source
  ↓
Glottal pulses / turbulence
  ↓
Vocal tract filter
  ↓
Speech signal
```

* `F0` / fundamental frequency mainly comes from the source.
* Formants come from resonances of the vocal tract.
* Voiced speech has periodic structure.
* Unvoiced speech is more noise-like.

### RMS + ZCR

Two simple features:

* RMS → signal energy
* ZCR → zero-crossing rate

They can provide a toy voiced/unvoiced heuristic.

⚠️ RMS + ZCR is NOT a production-quality VAD system.

Production systems can use pitch/spectral features or learned VAD models.

---

# 5. Mel Spectrogram

### Pipeline

```text
Waveform
   ↓
Windowing
   ↓
FFT / STFT
   ↓
Power Spectrum |X[k]|²
   ↓
Mel Filterbank
   ↓
Log
   ↓
Log-Mel Spectrogram
```

Typical speech configuration:

```text
Window     = ~25 ms
Hop        = ~10 ms
Mel bands  = 80
Sample rate = 16 kHz
Frequency range = 0–8 kHz
```

At a 10 ms hop:

```text
~100 frames / second
```

### Mel formula

```text
mel(f) = 2595 × log10(1 + f/700)
```

The mel scale provides greater resolution at lower frequencies and wider
bands at higher frequencies.

---

# 6. STFT: Window vs Hop

### Window length

Controls the time-frequency trade-off:

```text
Short window
→ better time resolution
→ worse frequency resolution

Long window
→ worse time resolution
→ better frequency resolution
```

This is fundamentally related to:

```text
Δt × Δf ≥ constant
```

### Why Hann window?

A rectangular window causes more spectral leakage.

Hann windowing reduces leakage and concentrates spectral energy.

---

# 7. Whisper Architecture

Whisper is an **encoder-decoder Transformer** operating on log-mel audio features.

```text
Audio
 ↓
80-band Log-Mel
 ↓
Convolutional Stem
 ↓
Transformer Encoder
 ↓
Audio Representation
 ↓
Transformer Decoder
 ↓
Text
```

### Encoder

The encoder processes the audio representation and builds an audio memory.

It is effectively bidirectional over the input audio frames.

### Decoder

The decoder is autoregressive:

```text
Previous tokens
      ↓
Causal self-attention
      +
Cross-attention → Audio encoder output
      ↓
Next token
      ↓
Append token
      ↓
Repeat
```

### Two important attentions

```text
Self-attention
→ attends to previous generated text tokens

Cross-attention
→ attends to encoded audio
```

---

# 8. Whisper Task Prefix

Whisper can use special tokens to control the task.

Conceptually:

```text
<start>
<language>
<transcribe / translate>
<timestamps setting>
```

The same model can therefore be directed toward transcription or translation.

Important:

```text
whisper-base.en
```

is English-focused.

For multilingual translation, use a multilingual Whisper checkpoint.

---

# 9. Word Error Rate — WER

WER is the standard metric emphasized in the notebook for ASR.

```text
WER = (S + I + D) / N
```

Where:

```text
S = substitutions
I = insertions
D = deletions
N = number of words in reference
```

Example:

```text
Reference:
"hello world"

Prediction:
"hello word"

1 substitution
2 reference words

WER = 1 / 2 = 50%
```

### Important

Normalize text before calculating WER:

```text
case
punctuation
formatting
```

Otherwise, you may measure formatting differences rather than recognition quality.

---

# 10. Long Audio + Chunking

Whisper was trained around **30-second windows**.

For longer audio:

```text
Long audio
   ↓
30 sec chunk
   ↓
5 sec overlap/stride
   ↓
30 sec chunk
   ↓
...
   ↓
Stitch predictions
```

Typical configuration:

```python
chunk_length_s = 30
stride_length_s = 5
return_timestamps = True
```

### Why overlap?

Suppose a word occurs exactly around the boundary:

```text
Chunk 1 | word
        |
        | Chunk boundary
        |
Chunk 2 | word
```

The overlap ensures the complete word can appear inside at least one chunk,
reducing boundary deletions.

---

# 11. CLAP

Whisper:

```text
Audio → Text
```

CLAP:

```text
Audio ───────┐
             ↓
       Shared embedding space
             ↑
Text ────────┘
```

Audio and text are mapped to vectors where semantic similarity can be measured.

Typically:

```text
audio embedding → L2 normalize
text embedding  → L2 normalize

similarity = dot(audio, text)
```

For normalized vectors:

```text
cosine similarity = dot product
```

Applications:

* Text → audio retrieval
* Audio → text retrieval
* Zero-shot audio classification
* Semantic search

---

# 🎥 VIDEO MODALITY

# 12. Video = 4-D Tensor

A video clip can be represented as:

```text
(T, H, W, C)
```

Where:

```text
T = number of frames
H = height
W = width
C = channels
```

Example:

```text
(T, 1080, 1920, 3)
```

### Raw memory

```text
Memory ≈ T × H × W × C × bytes_per_element
```

Example from notebook:

```text
10 minutes
30 FPS
1920 × 1080
RGB uint8

≈ 112 GB raw
```

Float32 would be approximately:

```text
≈ 448 GB
```

Therefore:

> You never feed raw long video directly into a model.

---

# 13. Video Codec / Random Access

Video is not simply a collection of independent JPEG images.

Common frame types:

```text
I-frame → complete image
P-frame → predicted from previous frames
B-frame → predicted using neighboring frames
```

These form a GOP (Group of Pictures).

Therefore:

```text
Random frame access
       ↓
May require decoding from previous keyframe
```

This matters for frame sampling and video preprocessing.

---

# 14. Video Redundancy

Video has two major forms of redundancy.

### Temporal redundancy

Adjacent frames are often very similar.

Can measure using:

```text
Mean Absolute Difference
SSIM
```

High SSIM between adjacent frames:

```text
Frame t ≈ Frame t+1
```

Therefore, processing every frame wastes compute.

### Spatial redundancy

Large image regions may contain little texture/detail.

Resizing and patchification remove some of this redundancy cheaply.

---

# 15. Frame Sampling

Three important approaches:

### 1. Uniform / fixed stride

```text
Frame 0
Frame k
Frame 2k
Frame 3k
...
```

Pros:

* Cheap
* Simple
* Reproducible

Cons:

* Can miss short events

### 2. Segment-based / TSN-style

Split video into `K` temporal segments:

```text
|---- segment 1 ----|
|---- segment 2 ----|
|---- segment 3 ----|
...
```

Sample one frame from each.

Advantage:

```text
Good temporal coverage with bounded frame count
```

### 3. Content-aware

Sample frames when some signal spikes:

```text
motion energy
scene changes
event scores
```

Better for rare events, but preprocessing is more expensive.

---

# 16. Why Frame-Only Models Fail

Suppose:

```text
Video A: object moving LEFT → RIGHT

Video B: object moving RIGHT → LEFT
```

The frames can be exactly the same set but in reverse order.

A model looking at one random frame cannot determine:

```text
direction
speed
temporal order
```

Therefore:

> Temporal information must be explicitly modeled.

---

# 17. Optical Flow

Optical flow estimates pixel displacement between consecutive frames.

For each pixel:

```text
(u, v)
```

where:

```text
u = horizontal displacement
v = vertical displacement
```

From flow:

```text
magnitude = sqrt(u² + v²)

angle = atan2(v, u)
```

Therefore:

```text
Magnitude → how much motion
Angle     → direction of motion
```

The notebook uses dense **Farneback optical flow**.

### Important limitation

Optical flow may capture camera motion as well as object motion.

If the camera moves:

```text
observed flow = object motion + camera motion
```

A possible correction:

```text
flow_corrected = flow - median(flow)
```

This estimates and removes common/global motion.

---

# 18. 3-D CNN vs 2-D CNN

### 2-D CNN

Processes frames independently:

```text
Frame 1 → CNN
Frame 2 → CNN
Frame 3 → CNN
...
```

Good:

```text
cheap
simple
```

Weakness:

```text
poor temporal modeling
```

### 3-D CNN

Kernel operates over:

```text
time × height × width
```

Example:

```text
k × k × k
```

It directly learns spatio-temporal features.

But cost grows approximately with:

```text
k³
```

and memory increases substantially with clip length.

---

# 19. (2+1)D Convolution

Instead of:

```text
3-D convolution
```

factorize:

```text
2-D spatial convolution
        ↓
1-D temporal convolution
```

Kernel footprint:

```text
Full 3-D:
k³

(2+1)D:
k² + k
```

Real parameter counts:

```text
Full 3-D:
k_t × k_h × k_w × C_in × C_out

(2+1)D:
(k_h × k_w × C_in × C_mid)
+
(k_t × C_mid × C_out)
```

Important:

> (2+1)D is not mathematically identical to an arbitrary full 3-D convolution.
> It is a different factorized parameterization with a comparable receptive field.

---

# 20. Video Transformer Tokenization

Transformers require a sequence.

A video is divided into spatial patches.

For:

```text
T frames
H × W resolution
Patch size P
```

number of tokens:

```text
N = T × (H/P) × (W/P)
```

Each token contains a feature vector:

```text
(B, N, d)
```

where:

```text
B = batch size
N = token count
d = embedding dimension
```

### Why token count matters

Self-attention:

```text
O(N²)
```

Therefore, reducing `N` is one of the biggest levers for video Transformer
performance.

---

# 21. Tubelets

Instead of processing one spatial patch from one frame:

```text
P × P
```

process a small space-time cube:

```text
P × P × t
```

This is called a **tubelet**.

Increasing tubelet depth:

```text
t ↑
→ fewer tokens
→ lower attention cost
```

---

# 22. TimeSformer

TimeSformer uses **divided space-time attention**.

Instead of joint attention over all tokens:

```text
O(N²)
```

perform:

```text
Temporal attention
       ↓
Spatial attention
```

Complexity:

```text
O(T² × S) + O(S² × T)
```

where:

```text
T = temporal tokens
S = spatial tokens per frame
```

This reduces the cost compared with full joint space-time attention.

### Critical interview distinction

> VideoMAE does NOT use TimeSformer's divided attention.

---

# 23. Video Model Comparison

| Model                      | Main Idea                           | Strength                      | Weakness                       |
| -------------------------- | ----------------------------------- | ----------------------------- | ------------------------------ |
| 2-D CNN                    | CNN per frame                       | Cheap/simple                  | Weak temporal modeling         |
| 3-D CNN                    | Spatio-temporal kernels             | Local motion                  | Expensive                      |
| R(2+1)D                    | Spatial + temporal factorization    | Cheaper than 3-D              | Local receptive field          |
| TimeSformer                | Divided space/time attention        | Long-range temporal relations | Token cost                     |
| ViViT                      | Video Transformer + tubelets        | Flexible tokenization         | Compute-heavy                  |
| VideoMAE                   | Masked video pretraining + tubelets | Data-efficient representation | More complex training pipeline |
| CLIP4Clip / video-language | Shared video/text embeddings        | Retrieval/QA                  | Heavier infrastructure         |

---

# 24. VideoMAE

VideoMAE uses:

```text
Video
 ↓
Sample frames
 ↓
Tubelet tokenization
 ↓
Masked autoencoding
 ↓
Transformer encoder
 ↓
Learned representation
```

The notebook's guarded model:

```text
16 RGB frames
→ (1, 16, 3, 224, 224)
→ VideoMAE
→ logits
→ (1, 400)
```

The 400 outputs correspond to Kinetics-400 classes.

---

# 25. Video Classification Evaluation

The notebook intentionally creates an 8-class problem:

```text
4 directions × 2 speed bands
```

### Model A — frame-only

Input:

```text
random frame
```

Cannot reliably determine:

```text
direction
speed
```

Expected accuracy:

```text
≈ 12.5%
```

because:

```text
1 / 8 = 12.5%
```

### Model B — direction-only flow features

Features:

```text
mean dx
std dx
mean dy
std dy
```

Good for:

```text
direction
```

Bad for:

```text
speed
```

because flow direction features don't sufficiently encode magnitude.

### Model C — augmented temporal features

Adds:

```text
mean flow magnitude
std flow magnitude
temporal acceleration
```

Now the model gets information about:

```text
direction + speed
```

Accuracy improves substantially.

---

# 26. Honest Evaluation

Small datasets should not rely on one train/test split.

Notebook approach:

```text
10 random 70/30 splits
```

Report:

```text
mean ± standard deviation
min–max
```

Also inspect:

```text
confusion matrix
misclassified examples
```

### Interview principle

> A single accuracy number is not enough. Understand the failure mode and
> evaluate using a metric appropriate for the task.

---

# 27. Retrieval Metrics

For video-text retrieval, accuracy may not be the best metric.

Use:

```text
Recall@K
MRR
nDCG
```

Other video tasks:

| Task               | Appropriate Metrics                  |
| ------------------ | ------------------------------------ |
| Classification     | Accuracy / Macro-F1 / Top-K accuracy |
| Retrieval          | Recall@K / MRR / nDCG                |
| Localization       | Temporal IoU                         |
| Detection          | Event-level precision/recall         |
| Production serving | Latency / throughput / cost          |

---

# 28. IMPORTANT SHAPES TO MEMORIZE

## Audio

```text
Waveform:
(samples,)

Power spectrogram:
(1 + n_fft/2, frames)

Log-mel:
(80, frames)

Token IDs:
(seq_len,)

CLAP:
(512,)
```

## Video

```text
Raw frames:
(T, H, W, 3)

Sampled:
(T', H, W, 3)

Resized:
(T', h, w, 3)

CNN input:
(B, C, T', h, w)

ViT tokens:
(B, N, d)

N = T' × (h/P) × (w/P)

Tubelet tokens:
reduced by tubelet depth t

Optical flow:
(T'-1, H, W, 2)

Clip embedding:
(B, d)

Classifier logits:
(B, n_classes)
```

---

# 29. ⭐ TOP INTERVIEW QUESTIONS + ANSWERS

## Q1. Why do we convert audio into a mel spectrogram?

**Answer:**

Raw waveform contains amplitude over time, but neural models benefit from a
compact time-frequency representation. We use STFT to obtain frequency energy
over short windows, apply a mel filterbank to compress frequencies according
to human auditory perception, and take the log to compress dynamic range.
Whisper uses this log-mel representation as its encoder input.

---

## Q2. What is the difference between sampling rate and bit depth?

**Answer:**

Sampling rate controls how frequently we measure the signal and therefore
determines the maximum representable frequency through the Nyquist limit,
`fs/2`. Bit depth controls how precisely we represent the amplitude of each
sample and approximately adds 6 dB of dynamic range per bit.

---

## Q3. What is aliasing?

**Answer:**

Aliasing occurs when frequencies above the new Nyquist frequency are sampled
and appear as incorrect lower frequencies. Before downsampling, we therefore
apply an anti-alias low-pass filter.

---

## Q4. Why use a mel scale?

**Answer:**

The mel scale is perceptually motivated. Human hearing has finer frequency
resolution at lower frequencies and coarser resolution at higher frequencies.
A mel filterbank therefore compresses the linear FFT representation into
perceptually meaningful frequency bands.

---

## Q5. Explain Whisper's architecture.

**Answer:**

Whisper uses an encoder-decoder Transformer. The encoder consumes log-mel audio
features and produces an audio representation. The autoregressive decoder uses
causal self-attention over previously generated tokens and cross-attention to
the encoder's audio representation to generate the next token.

---

## Q6. What is the difference between self-attention and cross-attention in Whisper?

**Answer:**

Self-attention allows the decoder to look at previously generated text tokens.
Cross-attention allows the decoder to look at the encoded audio features.
Therefore, self-attention handles the generated sequence context while
cross-attention connects text generation to the audio input.

---

## Q7. How do you evaluate an ASR model?

**Answer:**

The notebook uses Word Error Rate:

```text
WER = (Substitutions + Insertions + Deletions) / Reference words
```

Before calculating it, the reference and prediction should be normalized so
case and punctuation differences don't artificially affect the metric.

---

## Q8. How does Whisper handle audio longer than 30 seconds?

**Answer:**

Whisper was trained around 30-second windows, so long audio is chunked into
approximately 30-second segments. Neighboring chunks can overlap, for example
with a 5-second stride, so words near boundaries are less likely to be lost.
The resulting predictions are then stitched together and timestamps can be
returned.

---

## Q9. What is CLAP and how is it different from Whisper?

**Answer:**

Whisper is primarily an automatic speech recognition model that converts speech
into text. CLAP maps audio and text into a shared embedding space, allowing
semantic similarity between audio and text. This enables text-to-audio
retrieval and zero-shot audio classification.

---

## Q10. Why is video much more expensive than images?

**Answer:**

Video introduces a temporal dimension. Instead of processing one `H × W × C`
image, we process `T × H × W × C`. Memory therefore grows linearly with the
number of frames, and Transformer attention can grow quadratically with the
number of video tokens.

---

## Q11. Why can't we process every video frame?

**Answer:**

Adjacent frames contain significant temporal redundancy. Processing every frame
increases compute and memory without necessarily adding much information.
Therefore we usually sample frames using uniform, segment-based, or
content-aware strategies.

---

## Q12. What is the difference between uniform and TSN-style sampling?

**Answer:**

Uniform sampling selects frames at fixed intervals. TSN-style sampling divides
the video into temporal segments and samples one frame from each segment.
TSN provides better temporal coverage while maintaining a fixed number of
frames.

---

## Q13. Why does a frame-only model fail for action recognition?

**Answer:**

A single frame does not encode temporal order. For example, opening and closing
a door can contain nearly identical frames but in opposite temporal order.
Speed also requires comparing motion over time. Therefore temporal information
must be modeled.

---

## Q14. What is optical flow?

**Answer:**

Optical flow estimates the apparent displacement of pixels between consecutive
frames. It produces a vector `(u,v)` for each pixel. The magnitude represents
how much the pixel moved and the angle represents the direction.

---

## Q15. What is a major limitation of optical flow?

**Answer:**

Optical flow measures apparent motion, not necessarily object motion. Camera
movement can create global flow across the entire image. A common correction is
to estimate global motion, for example using the median flow, and subtract it
before computing object-motion statistics.

---

## Q16. Compare 2-D CNN and 3-D CNN.

**Answer:**

A 2-D CNN processes each frame independently, so it is cheap but has weak
temporal modeling. A 3-D CNN applies kernels across time and space, allowing it
to learn motion directly, but its computational and memory cost is much higher.

---

## Q17. Why is (2+1)D convolution useful?

**Answer:**

It factorizes a 3-D convolution into spatial 2-D convolution followed by
temporal 1-D convolution. The kernel footprint changes from approximately
`k³` to `k²+k`, reducing the spatio-temporal cost while retaining a comparable
receptive field.

---

## Q18. Is (2+1)D mathematically equivalent to 3-D convolution?

**Answer:**

No. It is a different factorized parameterization. It can provide a comparable
receptive field but does not represent every arbitrary full 3-D kernel unless
additional assumptions such as separability hold.

---

## Q19. Why are video Transformers expensive?

**Answer:**

A video is converted into many tokens. If there are `N` tokens, standard
self-attention has approximately `O(N²)` complexity. Since time adds another
dimension, video can produce far more tokens than an image. Sampling, larger
patches, tubelets, and efficient attention are therefore important.

---

## Q20. What are tubelets?

**Answer:**

A tubelet is a small space-time cube, for example `P × P × t`, instead of a
single-frame `P × P` patch. By grouping multiple frames into one token, tubelets
reduce the number of tokens and therefore reduce Transformer attention cost.

---

## Q21. Explain TimeSformer.

**Answer:**

TimeSformer uses divided space-time attention. Instead of jointly attending
over all space-time tokens, it performs temporal attention followed by spatial
attention. The complexity becomes approximately:

```text
O(T²S) + O(S²T)
```

instead of full joint `O(N²)` attention.

---

## Q22. Does VideoMAE use divided space-time attention?

**Answer:**

No. This is an important distinction. VideoMAE uses tubelet tokenization and a
ViT-style encoder, with masked autoencoding used for pretraining. Divided
space-time attention is associated with TimeSformer.

---

## Q23. What is VideoMAE?

**Answer:**

VideoMAE is a masked autoencoder approach for video representation learning.
Video is converted into tubelet tokens, a large portion is masked, and the
model learns useful representations by reconstructing the missing content.
This makes self-supervised video pretraining more data-efficient.

---

## Q24. Why did the notebook use synthetic video data?

**Answer:**

The goal was to demonstrate the mechanics and evaluation methodology
reproducibly without depending on downloads or a large labeled dataset. The
synthetic data gives controlled labels such as direction and speed, allowing
us to explicitly demonstrate where frame-only features fail and temporal
features fix the problem.

---

## Q25. Why was the frame-only classifier expected to achieve ~12.5%?

**Answer:**

There are 8 classes:

```text
4 directions × 2 speed bands = 8
```

Random guessing therefore gives:

```text
1 / 8 = 12.5%
```

The point is to demonstrate that a single frame lacks enough temporal
information to reliably identify direction and speed.

---

## Q26. Why did direction-only optical-flow features fail to distinguish speed?

**Answer:**

The features were:

```text
mean dx
std dx
mean dy
std dy
```

They capture the sign and variability of movement, so direction is represented,
but they do not explicitly provide flow magnitude. Therefore slow and fast
examples within the same direction remain difficult to distinguish.

---

## Q27. How did Model C improve the classifier?

**Answer:**

Model C added:

```text
mean flow magnitude
std flow magnitude
temporal acceleration
```

These features provide information about motion magnitude and changes in motion,
so the model can distinguish both direction and speed.

---

## Q28. Why use multiple train/test splits?

**Answer:**

The dataset is small, so one train/test split could give a misleading result.
The notebook evaluates each model over 10 random 70/30 splits and reports
mean ± standard deviation and min-max. A representative split is also used
for confusion matrices and qualitative error analysis.

---

## Q29. Which metric would you use for video retrieval?

**Answer:**

For retrieval I would use Recall@K, MRR, or nDCG rather than simple
classification accuracy. The metric should match the actual task.

---

## Q30. How would you optimize a video Transformer that is too slow/OOM?

**Answer:**

First reduce the token count:

```text
fewer frames
larger spatial patches
larger tubelet depth
smaller resolution
```

I could also use efficient/divided attention such as the TimeSformer approach.
The key principle is to control `N`, because attention cost is approximately
`O(N²)`.

---

# 30. 🔥 RAPID-FIRE NUMBERS TO MEMORIZE

```text
Speech:
16 kHz sample rate
8 kHz Nyquist
~6 dB / bit
~25 ms window
~10 ms hop
80 mel bands
~100 mel frames/sec
Whisper ≈ 30 sec native window
5 sec overlap example

Video:
Video tensor = (T,H,W,C)
CNN input = (B,C,T,H,W)
ViT tokens = (B,N,d)
N = T × (H/P) × (W/P)
Attention = O(N²)

Optical flow:
(u,v)
magnitude = sqrt(u²+v²)

3-D convolution:
k³

(2+1)D:
k²+k

TimeSformer:
O(T²S + S²T)

8-class synthetic classifier:
4 directions × 2 speed bands
chance = 12.5%

VideoMAE demo:
16 frames
224×224
input ≈ (1,16,3,224,224)
Kinetics-400 logits ≈ (1,400)
```

# 31. 🎯 60-SECOND INTERVIEW SUMMARY

If asked to summarize the notebook:

> "The notebook covers how ML models process speech and video as tensors rather
> than raw human-perceived signals. For speech, the pipeline goes from waveform
> through sampling, STFT, mel filtering and log compression to an 80-band
> log-mel representation, which Whisper processes with an encoder-decoder
> Transformer. Whisper uses causal self-attention for generated tokens and
> cross-attention to the audio representation, while ASR quality is evaluated
> using normalized WER. For video, the key challenge is the additional temporal
> dimension, which increases memory and Transformer token count. We therefore
> sample frames, use temporal features such as optical flow, and can model video
> using 3-D CNNs, factorized (2+1)D convolutions, or video Transformers such as
> TimeSformer, ViViT and VideoMAE. The central engineering principle is to
> control tensor shape, token count, temporal information and evaluation
> methodology according to the latency, accuracy and task requirements."

# 32. ⭐ FINAL MEMORY MAP

```text
SPEECH
│
├── Waveform
│   ├── Sampling → fs/2 → Nyquist
│   └── Bit depth → ~6 dB/bit
│
├── STFT
│   ├── Window → time/frequency trade-off
│   └── Hann → reduce leakage
│
├── Mel
│   └── 80-band log-mel
│
├── Whisper
│   ├── Encoder → audio representation
│   ├── Decoder → autoregressive text
│   ├── Self-attention → previous tokens
│   └── Cross-attention → audio
│
├── Evaluation
│   └── WER = (S+I+D)/N
│
└── CLAP
    └── Shared audio-text embedding space


VIDEO
│
├── Tensor
│   └── (T,H,W,C)
│
├── Redundancy
│   ├── Temporal
│   └── Spatial
│
├── Sampling
│   ├── Uniform
│   ├── TSN / segments
│   └── Content-aware
│
├── Motion
│   └── Optical Flow (u,v)
│
├── CNN
│   ├── 2-D → frame-level
│   ├── 3-D → space + time
│   └── (2+1)D → spatial + temporal factorization
│
├── Transformer
│   ├── Patches
│   ├── Tubelets
│   └── Attention O(N²)
│
├── Architectures
│   ├── TimeSformer → divided space/time attention
│   ├── ViViT → factorized video Transformer + tubelets
│   └── VideoMAE → masked pretraining + tubelets
│
└── Evaluation
    ├── Classification → Accuracy / Macro-F1
    ├── Retrieval → Recall@K / MRR / nDCG
    ├── Localization → Temporal IoU
    └── Production → Latency / Throughput / Cost
```

## ⭐ Interview Golden Rules

1. **Always talk about tensor shapes.**
2. **For audio, remember waveform → STFT → mel → log-mel → model.**
3. **For video, remember that time multiplies memory and token count.**
4. **If someone asks why sampling is needed, answer: redundancy + compute.**
5. **If someone asks why frame-only fails, answer: no temporal order.**
6. **If someone asks about optical flow, mention both object motion AND camera motion.**
7. **If someone asks about Transformers, immediately discuss token count and `O(N²)`.**
8. **Know the distinction: TimeSformer ≠ VideoMAE.**
9. **Know why tubelets reduce video token count.**
10. **Never quote an accuracy without explaining the evaluation setup and failure mode.**

```
```
